# week 1 - structured output for the docs rag

## rag recap

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 382 chunks from 95 documents


In [2]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [3]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [4]:
import os
from openai import OpenAI

# groq, through the openai client
# free tier is 8k tokens/min, max_retries makes it wait and retry on 429
openai_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    max_retries=10,
)

MODEL = "openai/gpt-oss-20b"

In [5]:
def llm(user_prompt, instructions=None, model=MODEL):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [6]:
answer = rag('how do I implement LLM as a judge?')
print(answer[:500])

**Short answer**

1. Create a toy / real‑world dataset that contains  
   * the question / prompt that you send to your LLM app,  
   * the *target* (ground‑truth) response,  
   * the *new* (model‑generated) response,  
   * and, if you want to benchmark the judge, a manual “golden” label.

2. Write a prompt that asks an LLM to evaluate the *new* response.  
   * For a **reference‑based** judge you pass both *target* and *new* to the prompt.  
   * For an **open‑ended** judge you only pass the 


## llm_structured

In [7]:
def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model=MODEL,
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

calendar event from the previous lesson

In [8]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)
response

CalendarEvent(name='science fair', date='Friday', participants=['Alice', 'Bob'])

## structured rag

In [9]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [10]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [11]:
answer = rag_structured('how do I do llm evals?')

print(answer.answer[:100])
print(answer.found_answer)

To run LLM‑based evaluations (LLM evals) in Evidently:
1. **Install** the LLM extras: `pip install e
True


In [12]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)

The provided documentation does not contain information on installing Kafka on Windows.
False


### optional answer

In [13]:
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

RAGResponse.model_json_schema()

{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [14]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [15]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


### docstrings and field descriptions

In [16]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool

RAGResponse.model_json_schema()

{'description': "The response from the documentation RAG system\n\nIf the answer to the question wasn't found in the database, `answer` is None",
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [17]:
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [18]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


### more fields

In [19]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [20]:
def show(r):
    print(r.answer[:100])
    print(r.found_answer)
    print(r.confidence)
    print(r.confidence_explanation)
    print(r.answer_type)
    print(r.followup_questions)

answer = rag_structured('how do I evaluate llms', RAGResponse)
show(answer)

### How to Evaluate LLMs with an LLM Jury

1. **Install the required libraries**
   ```bash
   pip i
True
0.95
The answer uses only facts present in the context and fully addresses the user's question.
how-to
['How can I customize the prompt to evaluate other aspects like factuality or creativity?', 'Can I use self‑hosted models as judges, and how would I set them up?', 'What metrics are available in the `Report` class for summarizing results?', 'How do I upload results to Evidently Cloud for collaboration?']


In [21]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)
show(answer)

I couldn't find any information on installing Kafka on Windows in the provided documentation.
False
0.1
The context only covers the installation of the Evidently Python package, not Kafka. No mention of Kafka installation steps was found.
explanation
['Is there any other documentation you would like me to check for Kafka installation steps?', 'Do you need instructions on how to install Kafka from the official site or via a package manager like Chocolatey?', 'Would you like guidance on verifying a Kafka installation once it is installed?']


### nested, answer or answer_not_found

In [22]:
from pydantic import model_validator

class AnswerNotFound(BaseModel):
    explanation: str

class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [23]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The documentation does not contain information on installing Kafka on Windows.'), found_answer=False, answer=None)

In [24]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='To run LLM evaluations with Evidently, follow these steps:\n\n1. **Compute descriptors** – Build a `Dataset` that includes the text fields you want to evaluate and any custom or built‑in descriptors (e.g., `TextLength`, `SentenceCount`, `LLMAssessment`, etc.).\n\n```python\nfrom evidently.future.datasets import Dataset, DataDefinition\nfrom evidently.future.descriptors import TextLength, SentenceCount\n\nref_dataset = Dataset.from_pandas(\n    pd.DataFrame(ref_data),\n    data_definition=DataDefinition(),\n    descriptors=[\n        TextLength("target_response", alias="Length"),\n        SentenceCount("target_response", alias="Sentence"),\n    ],\n)\n```\n\n2. **Create a report** – Use the `TextEvals` preset (which runs all descriptor metrics) and optionally enable tests.\n\n```python\nfrom evidently.future.presets import TextEvals\nfrom evidently.future.report import Report\n\nreport = Report([\n    Te

### validation errors

In [25]:
from pydantic import ValidationError

try:
    AnswerResponse(found_answer=False)
except ValidationError as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for AnswerResponse
  Value error, Provide either 'answer' or 'answer_not_found'. [type=value_error, input_value={'found_answer': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
